In [1]:
# ============================================================
# MEDICORE HEALTHCARE ANALYTICS
# FACT APPOINTMENTS
# CELL 1 — LOAD DIMENSIONS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

np.random.seed(42)

# Project location
BASE_DIR = Path(
    r"C:\Users\janak\MediCore_Healthcare_Analytics"
)

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
VALIDATION_DIR = BASE_DIR / "data" / "validation"

# ------------------------------------------------------------
# Load dimension tables
# ------------------------------------------------------------

dim_date = pd.read_csv(
    RAW_DIR / "dim_date.csv"
)

dim_hospital = pd.read_csv(
    RAW_DIR / "dim_hospital.csv"
)

dim_department = pd.read_csv(
    RAW_DIR / "dim_department.csv"
)

dim_doctor = pd.read_csv(
    RAW_DIR / "dim_doctor.csv"
)

dim_patient = pd.read_csv(
    RAW_DIR / "dim_patient.csv"
)

dim_insurance = pd.read_csv(
    RAW_DIR / "dim_insurance.csv"
)

print("All dimension tables loaded successfully!")

print("\nDimension sizes:")

print("dim_date:", dim_date.shape)
print("dim_hospital:", dim_hospital.shape)
print("dim_department:", dim_department.shape)
print("dim_doctor:", dim_doctor.shape)
print("dim_patient:", dim_patient.shape)
print("dim_insurance:", dim_insurance.shape)

All dimension tables loaded successfully!

Dimension sizes:
dim_date: (731, 11)
dim_hospital: (8, 7)
dim_department: (44, 5)
dim_doctor: (600, 9)
dim_patient: (100000, 10)
dim_insurance: (8, 6)


In [2]:
# ============================================================
# CELL 2 — APPOINTMENT IDs & DATES
# ============================================================

N_APPOINTMENTS = 500_000

# ------------------------------------------------------------
# Generate appointment IDs
# ------------------------------------------------------------

appointment_ids = np.array([
    f"A{i:06d}"
    for i in range(1, N_APPOINTMENTS + 1)
])

# ------------------------------------------------------------
# Convert date table
# ------------------------------------------------------------

dim_date["full_date"] = pd.to_datetime(
    dim_date["full_date"]
)

# ------------------------------------------------------------
# Generate appointment dates
# ------------------------------------------------------------

appointment_dates = np.random.choice(
    dim_date["full_date"].values,
    size=N_APPOINTMENTS
)

# ------------------------------------------------------------
# Create initial dataframe
# ------------------------------------------------------------

fact_appointments = pd.DataFrame({
    "appointment_id": appointment_ids,
    "appointment_date": appointment_dates
})

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Appointments created:", len(fact_appointments))
print("Columns:", fact_appointments.columns.tolist())

print(
    "\nDate range:",
    fact_appointments["appointment_date"].min(),
    "to",
    fact_appointments["appointment_date"].max()
)

print(
    "\nDuplicate appointment IDs:",
    fact_appointments["appointment_id"].duplicated().sum()
)

display(fact_appointments.head())

Appointments created: 500000
Columns: ['appointment_id', 'appointment_date']

Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00

Duplicate appointment IDs: 0


,appointment_id,appointment_date
0,A000001,2024-04-12
1,A000002,2025-03-11
2,A000003,2024-09-27
3,A000004,2024-04-16
4,A000005,2024-03-12


In [3]:
# ============================================================
# CELL 3 — ASSIGN PATIENTS
# ============================================================

# Randomly select patients from the patient master table
patient_indices = np.random.randint(
    0,
    len(dim_patient),
    size=N_APPOINTMENTS
)

fact_appointments["patient_id"] = (
    dim_patient.iloc[patient_indices]["patient_id"]
    .values
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Patients assigned successfully!")

print(
    "Unique patients used:",
    fact_appointments["patient_id"].nunique()
)

print(
    "Missing patient IDs:",
    fact_appointments["patient_id"].isna().sum()
)

print("\nSample:")

display(
    fact_appointments[
        ["appointment_id", "appointment_date", "patient_id"]
    ].head(10)
)

Patients assigned successfully!
Unique patients used: 99288
Missing patient IDs: 0

Sample:


,appointment_id,appointment_date,patient_id
0,A000001,2024-04-12,P079051
1,A000002,2025-03-11,P044247
2,A000003,2024-09-27,P017387
3,A000004,2024-04-16,P012971
4,A000005,2024-03-12,P044862
5,A000006,2025-12-01,P047087
6,A000007,2024-01-21,P045149
7,A000008,2025-09-06,P051628
8,A000009,2024-05-01,P007996
9,A000010,2025-04-11,P036277


In [4]:
# ============================================================
# CELL 4 — ASSIGN DOCTORS
# ============================================================

# Select random doctors
doctor_indices = np.random.randint(
    0,
    len(dim_doctor),
    size=N_APPOINTMENTS
)

selected_doctors = dim_doctor.iloc[doctor_indices].reset_index(drop=True)

# Assign doctor
fact_appointments["doctor_id"] = (
    selected_doctors["doctor_id"].values
)

# Assign department from the selected doctor
fact_appointments["department_id"] = (
    selected_doctors["department_id"].values
)

# Assign hospital from the selected doctor
fact_appointments["hospital_id"] = (
    selected_doctors["hospital_id"].values
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Doctors assigned successfully!")

print(
    "Unique doctors used:",
    fact_appointments["doctor_id"].nunique()
)

print(
    "Missing doctor IDs:",
    fact_appointments["doctor_id"].isna().sum()
)

print(
    "Missing department IDs:",
    fact_appointments["department_id"].isna().sum()
)

print(
    "Missing hospital IDs:",
    fact_appointments["hospital_id"].isna().sum()
)

print("\nSample:")

display(
    fact_appointments[
        [
            "appointment_id",
            "patient_id",
            "doctor_id",
            "department_id",
            "hospital_id"
        ]
    ].head(10)
)

Doctors assigned successfully!
Unique doctors used: 600
Missing doctor IDs: 0
Missing department IDs: 0
Missing hospital IDs: 0

Sample:


,appointment_id,patient_id,doctor_id,department_id,hospital_id
0,A000001,P079051,DR0151,D011,H002
1,A000002,P044247,DR0325,D035,H006
2,A000003,P017387,DR0265,D034,H006
3,A000004,P012971,DR0148,D006,H001
4,A000005,P044862,DR0171,D018,H004
5,A000006,P047087,DR0073,D003,H001
6,A000007,P045149,DR0182,D030,H005
7,A000008,P051628,DR0029,D028,H005
8,A000009,P007996,DR0329,D031,H006
9,A000010,P036277,DR0291,D020,H004


In [5]:
# ============================================================
# CELL 5 — APPOINTMENT TYPE & BOOKING CHANNEL
# ============================================================

# ------------------------------------------------------------
# Appointment type
# ------------------------------------------------------------

fact_appointments["appointment_type"] = np.random.choice(
    [
        "Consultation",
        "Follow-up",
        "Diagnostic"
    ],
    size=N_APPOINTMENTS,
    p=[0.60, 0.25, 0.15]
)

# ------------------------------------------------------------
# Booking channel
# ------------------------------------------------------------

fact_appointments["booking_channel"] = np.random.choice(
    [
        "Online",
        "Phone",
        "Walk-in",
        "Referral"
    ],
    size=N_APPOINTMENTS,
    p=[0.40, 0.25, 0.20, 0.15]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Appointment type distribution:")
display(
    fact_appointments["appointment_type"]
    .value_counts()
    .rename_axis("appointment_type")
    .reset_index(name="count")
)

print("\nBooking channel distribution:")
display(
    fact_appointments["booking_channel"]
    .value_counts()
    .rename_axis("booking_channel")
    .reset_index(name="count")
)

Appointment type distribution:


,appointment_type,count
0,Consultation,299924
1,Follow-up,124772
2,Diagnostic,75304



Booking channel distribution:


,booking_channel,count
0,Online,199915
1,Phone,125518
2,Walk-in,100260
3,Referral,74307


In [6]:
# ============================================================
# CELL 6 — APPOINTMENT LEAD TIME
# ============================================================

# Generate realistic booking lead times
# Most appointments are booked relatively close to the visit date.

lead_time = np.random.gamma(
    shape=2.2,
    scale=4.5,
    size=N_APPOINTMENTS
)

# Convert to whole days
lead_time = np.round(lead_time).astype(int)

# Keep lead time within realistic limits
lead_time = np.clip(
    lead_time,
    0,
    60
)

fact_appointments["lead_time_days"] = lead_time

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Lead time statistics:")

display(
    fact_appointments["lead_time_days"]
    .describe()
    .to_frame()
)

print(
    "\nMinimum lead time:",
    fact_appointments["lead_time_days"].min()
)

print(
    "Maximum lead time:",
    fact_appointments["lead_time_days"].max()
)

print(
    "Appointments booked same day:",
    (
        fact_appointments["lead_time_days"] == 0
    ).sum()
)

print("\nSample:")

display(
    fact_appointments[
        [
            "appointment_id",
            "appointment_date",
            "appointment_type",
            "booking_channel",
            "lead_time_days"
        ]
    ].head(10)
)

Lead time statistics:


,lead_time_days
count,500000.000000
mean,9.893856
std,6.671828
min,0.000000
25%,5.000000
50%,8.000000
75%,13.000000
max,60.000000



Minimum lead time: 0
Maximum lead time: 60
Appointments booked same day: 1515

Sample:


,appointment_id,appointment_date,appointment_type,booking_channel,lead_time_days
0,A000001,2024-04-12,Consultation,Referral,7
1,A000002,2025-03-11,Diagnostic,Referral,15
2,A000003,2024-09-27,Consultation,Online,10
3,A000004,2024-04-16,Consultation,Online,2
4,A000005,2024-03-12,Follow-up,Phone,5
5,A000006,2025-12-01,Diagnostic,Phone,9
6,A000007,2024-01-21,Consultation,Referral,1
7,A000008,2025-09-06,Consultation,Online,8
8,A000009,2024-05-01,Consultation,Online,14
9,A000010,2025-04-11,Follow-up,Online,10


In [7]:
# ============================================================
# CELL 7 — APPOINTMENT STATUS & NO-SHOW BEHAVIOR
# ============================================================

# ------------------------------------------------------------
# Base no-show probability
# ------------------------------------------------------------

no_show_probability = np.full(
    N_APPOINTMENTS,
    0.08
)

# ------------------------------------------------------------
# Lead time effect
# Longer lead time → slightly higher no-show probability
# ------------------------------------------------------------

no_show_probability += np.where(
    fact_appointments["lead_time_days"] >= 15,
    0.04,
    0
)

no_show_probability += np.where(
    fact_appointments["lead_time_days"] >= 30,
    0.03,
    0
)

# ------------------------------------------------------------
# Appointment type effect
# ------------------------------------------------------------

no_show_probability += np.where(
    fact_appointments["appointment_type"] == "Follow-up",
    0.02,
    0
)

no_show_probability -= np.where(
    fact_appointments["appointment_type"] == "Diagnostic",
    0.01,
    0
)

# ------------------------------------------------------------
# Booking channel effect
# ------------------------------------------------------------

no_show_probability += np.where(
    fact_appointments["booking_channel"] == "Online",
    0.02,
    0
)

no_show_probability -= np.where(
    fact_appointments["booking_channel"] == "Referral",
    0.01,
    0
)

# Keep probabilities within realistic limits
no_show_probability = np.clip(
    no_show_probability,
    0.03,
    0.25
)

# ------------------------------------------------------------
# Cancellation probability
# ------------------------------------------------------------

cancellation_probability = np.full(
    N_APPOINTMENTS,
    0.08
)

# Longer lead time → slightly higher cancellation probability
cancellation_probability += np.where(
    fact_appointments["lead_time_days"] >= 20,
    0.02,
    0
)

# Walk-in appointments are less likely to be cancelled
cancellation_probability -= np.where(
    fact_appointments["booking_channel"] == "Walk-in",
    0.02,
    0
)

cancellation_probability = np.clip(
    cancellation_probability,
    0.04,
    0.15
)

# ------------------------------------------------------------
# Generate random values
# ------------------------------------------------------------

random_values = np.random.random(N_APPOINTMENTS)

# First determine cancellations
is_cancelled = (
    random_values < cancellation_probability
)

# Generate another random value for remaining appointments
remaining_random = np.random.random(N_APPOINTMENTS)

is_no_show = (
    (~is_cancelled)
    &
    (remaining_random < no_show_probability)
)

# Everything else is completed
is_completed = (
    ~is_cancelled
    &
    ~is_no_show
)

# ------------------------------------------------------------
# Assign status
# ------------------------------------------------------------

fact_appointments["status"] = np.select(
    [
        is_cancelled,
        is_no_show,
        is_completed
    ],
    [
        "Cancelled",
        "No-show",
        "Completed"
    ]
)

# ------------------------------------------------------------
# No-show flag
# ------------------------------------------------------------

fact_appointments["no_show"] = np.where(
    fact_appointments["status"] == "No-show",
    1,
    0
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Appointment status distribution:")

status_summary = (
    fact_appointments["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="appointment_count")
)

status_summary["percentage"] = (
    status_summary["appointment_count"]
    / N_APPOINTMENTS
    * 100
)

display(status_summary)

print(
    "\nTotal appointments:",
    len(fact_appointments)
)

print(
    "No-show records:",
    fact_appointments["no_show"].sum()
)

print(
    "No-show rate:",
    round(
        fact_appointments["no_show"].mean() * 100,
        2
    ),
    "%"
)

TypeError: Choicelist and default value do not have a common dtype: The DType <class 'numpy.dtypes._PyLongDType'> could not be promoted by <class 'numpy.dtypes.StrDType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes._PyLongDType'>)

In [8]:
# ============================================================
# CELL 7 FIX — ASSIGN STATUS
# ============================================================

fact_appointments["status"] = np.select(
    [
        is_cancelled,
        is_no_show,
        is_completed
    ],
    [
        "Cancelled",
        "No-show",
        "Completed"
    ],
    default="Completed"
)

# ------------------------------------------------------------
# No-show flag
# ------------------------------------------------------------

fact_appointments["no_show"] = np.where(
    fact_appointments["status"] == "No-show",
    1,
    0
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Appointment status distribution:")

status_summary = (
    fact_appointments["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="appointment_count")
)

status_summary["percentage"] = (
    status_summary["appointment_count"]
    / N_APPOINTMENTS
    * 100
)

display(status_summary)

print(
    "\nTotal appointments:",
    len(fact_appointments)
)

print(
    "No-show records:",
    fact_appointments["no_show"].sum()
)

print(
    "No-show rate:",
    round(
        fact_appointments["no_show"].mean() * 100,
        2
    ),
    "%"
)

Appointment status distribution:


,status,appointment_count,percentage
0,Completed,415965,83.1930
1,No-show,45266,9.0532
2,Cancelled,38769,7.7538



Total appointments: 500000
No-show records: 45266
No-show rate: 9.05 %


In [9]:
# ============================================================
# CELL 8 — CANCELLATION REASONS
# ============================================================

cancellation_reasons = [
    "Patient Request",
    "Schedule Conflict",
    "Doctor Unavailable",
    "Transportation Issue",
    "Insurance Issue"
]

# Start with all values missing
fact_appointments["cancellation_reason"] = None

# Identify cancelled appointments
cancelled_mask = (
    fact_appointments["status"] == "Cancelled"
)

cancelled_count = cancelled_mask.sum()

# Generate reasons for cancelled appointments
generated_reasons = np.random.choice(
    cancellation_reasons,
    size=cancelled_count,
    p=[0.35, 0.25, 0.15, 0.15, 0.10]
)

fact_appointments.loc[
    cancelled_mask,
    "cancellation_reason"
] = generated_reasons

# ------------------------------------------------------------
# Intentional data-quality issue
# ------------------------------------------------------------
# A small percentage of cancelled appointments
# have missing cancellation reasons.

cancelled_indices = fact_appointments.index[
    cancelled_mask
]

missing_reason_count = int(
    len(cancelled_indices) * 0.05
)

missing_reason_indices = np.random.choice(
    cancelled_indices,
    size=missing_reason_count,
    replace=False
)

fact_appointments.loc[
    missing_reason_indices,
    "cancellation_reason"
] = None

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Cancellation reason distribution:")

reason_summary = (
    fact_appointments.loc[
        cancelled_mask,
        "cancellation_reason"
    ]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("cancellation_reason")
    .reset_index(name="count")
)

reason_summary["percentage"] = (
    reason_summary["count"]
    / cancelled_count
    * 100
)

display(reason_summary)

print(
    "\nTotal cancelled appointments:",
    cancelled_count
)

print(
    "Missing cancellation reasons:",
    fact_appointments.loc[
        cancelled_mask,
        "cancellation_reason"
    ].isna().sum()
)

print(
    "Non-cancelled appointments with reason:",
    (
        (~cancelled_mask)
        &
        fact_appointments["cancellation_reason"].notna()
    ).sum()
)

Cancellation reason distribution:


,cancellation_reason,count,percentage
0,Patient Request,12739,32.858727
1,Schedule Conflict,9389,24.217803
2,Doctor Unavailable,5512,14.217545
3,Transportation Issue,5450,14.057623
4,Insurance Issue,3741,9.649462
5,MISSING,1938,4.998839



Total cancelled appointments: 38769
Missing cancellation reasons: 1938
Non-cancelled appointments with reason: 0


In [10]:
# ============================================================
# CELL 9 — SCHEDULED DATETIME
# ============================================================

# Generate appointment time in minutes from midnight
appointment_minutes = np.random.randint(
    8 * 60,       # 08:00
    20 * 60,      # 20:00
    size=N_APPOINTMENTS
)

# Create scheduled datetime
fact_appointments["scheduled_datetime"] = (
    fact_appointments["appointment_date"]
    + pd.to_timedelta(
        appointment_minutes,
        unit="m"
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Scheduled datetime created successfully!")

print(
    "\nMinimum scheduled datetime:",
    fact_appointments["scheduled_datetime"].min()
)

print(
    "Maximum scheduled datetime:",
    fact_appointments["scheduled_datetime"].max()
)

print(
    "\nMissing scheduled datetime:",
    fact_appointments["scheduled_datetime"].isna().sum()
)

print("\nSample:")

display(
    fact_appointments[
        [
            "appointment_id",
            "appointment_date",
            "scheduled_datetime",
            "appointment_type",
            "booking_channel",
            "status"
        ]
    ].head(10)
)

Scheduled datetime created successfully!

Minimum scheduled datetime: 2024-01-01 08:00:00
Maximum scheduled datetime: 2025-12-31 19:59:00

Missing scheduled datetime: 0

Sample:


,appointment_id,appointment_date,scheduled_datetime,appointment_type,booking_channel,status
0,A000001,2024-04-12,2024-04-12 11:08:00,Consultation,Referral,Completed
1,A000002,2025-03-11,2025-03-11 16:21:00,Diagnostic,Referral,Completed
2,A000003,2024-09-27,2024-09-27 12:55:00,Consultation,Online,Completed
3,A000004,2024-04-16,2024-04-16 09:39:00,Consultation,Online,Completed
4,A000005,2024-03-12,2024-03-12 17:27:00,Follow-up,Phone,Completed
5,A000006,2025-12-01,2025-12-01 09:09:00,Diagnostic,Phone,Completed
6,A000007,2024-01-21,2024-01-21 19:09:00,Consultation,Referral,Completed
7,A000008,2025-09-06,2025-09-06 19:42:00,Consultation,Online,Completed
8,A000009,2024-05-01,2024-05-01 14:57:00,Consultation,Online,Completed
9,A000010,2025-04-11,2025-04-11 08:54:00,Follow-up,Online,Completed


In [11]:
# ============================================================
# CELL 10 — ASSIGN INSURANCE
# ============================================================

# Map each patient to their insurance provider
patient_insurance_map = (
    dim_patient
    .set_index("patient_id")["insurance_id"]
    .to_dict()
)

# Assign insurance based on patient
fact_appointments["insurance_id"] = (
    fact_appointments["patient_id"]
    .map(patient_insurance_map)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Insurance assignment completed!")

print(
    "\nMissing insurance IDs:",
    fact_appointments["insurance_id"].isna().sum()
)

print(
    "Unique insurance plans used:",
    fact_appointments["insurance_id"].nunique()
)

print("\nInsurance distribution:")

display(
    fact_appointments["insurance_id"]
    .value_counts()
    .sort_index()
)

Insurance assignment completed!

Missing insurance IDs: 70297
Unique insurance plans used: 8

Insurance distribution:


insurance_id
INS001    60201
INS002    60079
INS003    59563
INS004    49180
INS005    59348
INS006    54871
INS007    46041
INS008    40420
Name: count, dtype: int64

In [12]:
# ============================================================
# CELL 11 — PATIENT INSURANCE QUALITY VALIDATION
# ============================================================

# Count patients with and without insurance
patient_insurance_summary = (
    dim_patient["insurance_id"]
    .isna()
    .value_counts()
)

missing_patients = dim_patient["insurance_id"].isna().sum()
total_patients = len(dim_patient)
patients_with_insurance = total_patients - missing_patients

print("Patient Insurance Quality")
print("-" * 40)

print("Total patients:", total_patients)
print("Patients with insurance:", patients_with_insurance)
print("Patients without insurance:", missing_patients)

print(
    "Missing insurance %:",
    round((missing_patients / total_patients) * 100, 2)
)

# ------------------------------------------------------------
# Appointment impact
# ------------------------------------------------------------

missing_appointment_insurance = (
    fact_appointments["insurance_id"].isna().sum()
)

print("\nAppointment Impact")
print("-" * 40)

print("Total appointments:", len(fact_appointments))
print(
    "Appointments without insurance:",
    missing_appointment_insurance
)

print(
    "Appointment missing insurance %:",
    round(
        (missing_appointment_insurance / len(fact_appointments)) * 100,
        2
    )
)

Patient Insurance Quality
----------------------------------------
Total patients: 100000
Patients with insurance: 85932
Patients without insurance: 14068
Missing insurance %: 14.07

Appointment Impact
----------------------------------------
Total appointments: 500000
Appointments without insurance: 70297
Appointment missing insurance %: 14.06


In [13]:
# ============================================================
# CELL 12 — CREATE APPOINTMENT DATE ID
# ============================================================

# Create date → date_id lookup
date_id_map = (
    dim_date
    .set_index("full_date")["date_id"]
    .to_dict()
)

# Make sure appointment_date has the same date format
fact_appointments["appointment_date"] = pd.to_datetime(
    fact_appointments["appointment_date"]
).dt.normalize()

# Assign date_id
fact_appointments["appointment_date_id"] = (
    fact_appointments["appointment_date"]
    .map(date_id_map)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Appointment date_id created successfully!")

print(
    "\nMissing appointment_date_id:",
    fact_appointments["appointment_date_id"].isna().sum()
)

print(
    "Unique appointment dates:",
    fact_appointments["appointment_date"].nunique()
)

print(
    "Unique appointment date IDs:",
    fact_appointments["appointment_date_id"].nunique()
)

print("\nSample:")

display(
    fact_appointments[
        [
            "appointment_id",
            "appointment_date",
            "appointment_date_id",
            "scheduled_datetime",
            "status"
        ]
    ].head(10)
)

Appointment date_id created successfully!

Missing appointment_date_id: 0
Unique appointment dates: 731
Unique appointment date IDs: 731

Sample:


,appointment_id,appointment_date,appointment_date_id,scheduled_datetime,status
0,A000001,2024-04-12,20240412,2024-04-12 11:08:00,Completed
1,A000002,2025-03-11,20250311,2025-03-11 16:21:00,Completed
2,A000003,2024-09-27,20240927,2024-09-27 12:55:00,Completed
3,A000004,2024-04-16,20240416,2024-04-16 09:39:00,Completed
4,A000005,2024-03-12,20240312,2024-03-12 17:27:00,Completed
5,A000006,2025-12-01,20251201,2025-12-01 09:09:00,Completed
6,A000007,2024-01-21,20240121,2024-01-21 19:09:00,Completed
7,A000008,2025-09-06,20250906,2025-09-06 19:42:00,Completed
8,A000009,2024-05-01,20240501,2024-05-01 14:57:00,Completed
9,A000010,2025-04-11,20250411,2025-04-11 08:54:00,Completed


In [14]:
# ============================================================
# CELL 13 — PATIENT REGISTRATION DATE VALIDATION
# ============================================================

# Map patient registration dates
patient_registration_map = (
    dim_patient
    .set_index("patient_id")["registration_date"]
    .to_dict()
)

# Add registration date to appointments temporarily
fact_appointments["patient_registration_date"] = (
    fact_appointments["patient_id"]
    .map(patient_registration_map)
)

# Check appointments occurring before registration
before_registration = (
    fact_appointments["appointment_date"]
    < fact_appointments["patient_registration_date"]
)

print("Patient Registration Date Validation")
print("-" * 45)

print(
    "Appointments before patient registration:",
    before_registration.sum()
)

print(
    "Appointments on/after registration:",
    (~before_registration).sum()
)

print(
    "Missing patient registration dates:",
    fact_appointments["patient_registration_date"].isna().sum()
)

print(
    "\nInvalid appointment percentage:",
    round(
        before_registration.mean() * 100,
        2
    ),
    "%"
)

Patient Registration Date Validation
---------------------------------------------
Appointments before patient registration: 250668
Appointments on/after registration: 249332
Missing patient registration dates: 0

Invalid appointment percentage: 50.13 %


In [15]:
# ============================================================
# CELL 14 — CORRECT APPOINTMENT DATES
# ============================================================

# Identify appointments before patient registration
invalid_mask = (
    fact_appointments["appointment_date"]
    < fact_appointments["patient_registration_date"]
)

invalid_count = invalid_mask.sum()

# Move invalid appointment dates to the patient's
# registration date
fact_appointments.loc[
    invalid_mask,
    "appointment_date"
] = fact_appointments.loc[
    invalid_mask,
    "patient_registration_date"
]

print("Appointment date correction completed!")

print(
    "\nAppointments corrected:",
    invalid_count
)

print(
    "Total appointments:",
    len(fact_appointments)
)

print(
    "Appointments before registration:",
    (
        fact_appointments["appointment_date"]
        < fact_appointments["patient_registration_date"]
    ).sum()
)

Appointment date correction completed!

Appointments corrected: 250668
Total appointments: 500000
Appointments before registration: 0


In [16]:
# ============================================================
# CELL 15 — REGENERATE SCHEDULED DATETIME
# ============================================================

# Generate new appointment times
appointment_minutes = np.random.randint(
    8 * 60,
    20 * 60,
    size=N_APPOINTMENTS
)

# Rebuild scheduled datetime using corrected appointment dates
fact_appointments["scheduled_datetime"] = (
    fact_appointments["appointment_date"]
    + pd.to_timedelta(
        appointment_minutes,
        unit="m"
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Scheduled datetime regenerated successfully!")

print(
    "\nMinimum scheduled datetime:",
    fact_appointments["scheduled_datetime"].min()
)

print(
    "Maximum scheduled datetime:",
    fact_appointments["scheduled_datetime"].max()
)

print(
    "\nMissing scheduled datetime:",
    fact_appointments["scheduled_datetime"].isna().sum()
)

Scheduled datetime regenerated successfully!

Minimum scheduled datetime: 2024-01-02 09:56:00
Maximum scheduled datetime: 2025-12-31 19:58:00

Missing scheduled datetime: 0


In [17]:
# ============================================================
# CELL 16 — REFRESH APPOINTMENT DATE ID
# ============================================================

# Rebuild date → date_id lookup
date_id_map = (
    dim_date
    .set_index("full_date")["date_id"]
    .to_dict()
)

# Reassign date IDs using corrected appointment dates
fact_appointments["appointment_date_id"] = (
    fact_appointments["appointment_date"]
    .map(date_id_map)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Appointment date_id refreshed successfully!")

print(
    "\nMissing appointment_date_id:",
    fact_appointments["appointment_date_id"].isna().sum()
)

print(
    "Unique appointment dates:",
    fact_appointments["appointment_date"].nunique()
)

print(
    "Unique appointment date IDs:",
    fact_appointments["appointment_date_id"].nunique()
)

Appointment date_id refreshed successfully!

Missing appointment_date_id: 0
Unique appointment dates: 730
Unique appointment date IDs: 730


In [18]:
# ============================================================
# CELL 17 — FINAL APPOINTMENT STRUCTURE CHECK
# ============================================================

expected_columns = [
    "appointment_id",
    "patient_id",
    "doctor_id",
    "department_id",
    "hospital_id",
    "insurance_id",
    "appointment_date_id",
    "scheduled_datetime",
    "appointment_type",
    "booking_channel",
    "lead_time_days",
    "status",
    "no_show",
    "cancellation_reason"
]

print("FACT APPOINTMENTS — FINAL STRUCTURE")
print("=" * 50)

print("Rows:", len(fact_appointments))
print("Columns:", len(fact_appointments.columns))

print("\nRequired columns:")
for col in expected_columns:
    print(
        f"{'✓' if col in fact_appointments.columns else '✗'} {col}"
    )

print("\nMissing values:")
display(
    fact_appointments[expected_columns].isna().sum()
)

print("\nStatus distribution:")
display(
    fact_appointments["status"].value_counts()
)

print("\nNo-show flag distribution:")
display(
    fact_appointments["no_show"].value_counts()
)

FACT APPOINTMENTS — FINAL STRUCTURE
Rows: 500000
Columns: 16

Required columns:
✓ appointment_id
✓ patient_id
✓ doctor_id
✓ department_id
✓ hospital_id
✓ insurance_id
✓ appointment_date_id
✓ scheduled_datetime
✓ appointment_type
✓ booking_channel
✓ lead_time_days
✓ status
✓ no_show
✓ cancellation_reason

Missing values:


appointment_id              0
patient_id                  0
doctor_id                   0
department_id               0
hospital_id                 0
insurance_id            70297
appointment_date_id         0
scheduled_datetime          0
appointment_type            0
booking_channel             0
lead_time_days              0
status                      0
no_show                     0
cancellation_reason    463169
dtype: int64


Status distribution:


status
Completed    415965
No-show       45266
Cancelled     38769
Name: count, dtype: int64


No-show flag distribution:


no_show
0    454734
1     45266
Name: count, dtype: int64

In [19]:
# ============================================================
# CELL 18 — STATUS & NO-SHOW CONSISTENCY
# ============================================================

# Check whether no-show flag matches appointment status
invalid_no_show_flags = (
    ((fact_appointments["status"] == "No-show") &
     (fact_appointments["no_show"] != 1))
    |
    ((fact_appointments["status"] != "No-show") &
     (fact_appointments["no_show"] != 0))
)

print("STATUS & NO-SHOW CONSISTENCY")
print("=" * 50)

print(
    "Inconsistent no-show flags:",
    invalid_no_show_flags.sum()
)

print(
    "No-show appointments:",
    (fact_appointments["status"] == "No-show").sum()
)

print(
    "no_show = 1:",
    (fact_appointments["no_show"] == 1).sum()
)

print("\nStatus vs No-show flag:")

display(
    pd.crosstab(
        fact_appointments["status"],
        fact_appointments["no_show"]
    )
)

STATUS & NO-SHOW CONSISTENCY
Inconsistent no-show flags: 0
No-show appointments: 45266
no_show = 1: 45266

Status vs No-show flag:


no_show,0,1
status,,
Cancelled,38769,0
Completed,415965,0
No-show,0,45266


In [20]:
# ============================================================
# CELL 19 — REMOVE TEMPORARY COLUMNS
# ============================================================

# Remove temporary validation/helper columns
fact_appointments = fact_appointments.drop(
    columns=[
        "patient_registration_date",
        "appointment_date"
    ]
)

print("Temporary columns removed successfully!")

print("\nFinal shape:")
print("Rows:", len(fact_appointments))
print("Columns:", len(fact_appointments.columns))

print("\nFinal columns:")
print(fact_appointments.columns.tolist())

Temporary columns removed successfully!

Final shape:
Rows: 500000
Columns: 14

Final columns:
['appointment_id', 'patient_id', 'doctor_id', 'department_id', 'hospital_id', 'appointment_type', 'booking_channel', 'lead_time_days', 'status', 'no_show', 'cancellation_reason', 'scheduled_datetime', 'insurance_id', 'appointment_date_id']


In [21]:
# ============================================================
# CELL 20 — SAVE FACT APPOINTMENTS
# ============================================================

output_path = "../data/raw/fact_appointments.csv"

fact_appointments.to_csv(
    output_path,
    index=False
)

print("fact_appointments.csv saved successfully!")

print("\nFile path:")
print(output_path)

print("\nFinal shape:")
print(fact_appointments.shape)

print("\nFile size:")
import os
print(
    round(
        os.path.getsize(output_path) / (1024 * 1024),
        2
    ),
    "MB"
)

fact_appointments.csv saved successfully!

File path:
../data/raw/fact_appointments.csv

Final shape:
(500000, 14)

File size:
49.99 MB


In [22]:
# ============================================================
# CELL 21 — RELOAD & FINAL FILE VALIDATION
# ============================================================

# Reload the actual CSV from disk
fact_appointments_check = pd.read_csv(
    "../data/raw/fact_appointments.csv"
)

print("FACT APPOINTMENTS — FILE VALIDATION")
print("=" * 55)

print("Rows:", len(fact_appointments_check))
print("Columns:", len(fact_appointments_check.columns))

print(
    "\nDuplicate appointment IDs:",
    fact_appointments_check["appointment_id"].duplicated().sum()
)

print(
    "Missing patient IDs:",
    fact_appointments_check["patient_id"].isna().sum()
)

print(
    "Missing doctor IDs:",
    fact_appointments_check["doctor_id"].isna().sum()
)

print(
    "Missing appointment date IDs:",
    fact_appointments_check["appointment_date_id"].isna().sum()
)

print(
    "Missing insurance IDs:",
    fact_appointments_check["insurance_id"].isna().sum()
)

print(
    "Missing scheduled datetime:",
    fact_appointments_check["scheduled_datetime"].isna().sum()
)

print("\nStatus distribution:")
display(
    fact_appointments_check["status"].value_counts()
)

print("\nFinal validation completed.")

FACT APPOINTMENTS — FILE VALIDATION
Rows: 500000
Columns: 14

Duplicate appointment IDs: 0
Missing patient IDs: 0
Missing doctor IDs: 0
Missing appointment date IDs: 0
Missing insurance IDs: 70297
Missing scheduled datetime: 0

Status distribution:


status
Completed    415965
No-show       45266
Cancelled     38769
Name: count, dtype: int64


Final validation completed.
